In [ ]:
import numpy as np 
from pathlib import Path
import csv
import os
import glob
import keras
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,Model
from tensorflow.keras.layers import Lambda,Layer,Dense,Conv1D,LSTM
import seaborn as sns
from sklearn.metrics import confusion_matrix,classification_report
from tensorflow.keras.callbacks import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from itertools import combinations
import time
from scipy.signal import fftconvolve
import re
import shutil
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False
import math
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

In [ ]:
# ===========  Read ============
t0 = time.time()
num_classes = 5
mic_num = 4

data_folder = 'Public_Audio_Mel'
drone_types = ['Avata_npy','M300_npy','Mavic2_npy','Mavic3_npy','Pham4_npy']
loca_suffixes = ['x.csv','y.csv','z.csv']
def extract_num(filename):
    m = re.search(r'spectrogram2_(\d+)\.npy',filename)
    return int(m.group(1)) if m else -1


def load_spectrogram(
    data_folder,
    drone_types,
    loca_suffixes,
    num_classes = 5,
    mic_num = 4
    ):

    print('Reading data')
    all_data = []
    all_labels = []
    all_positions = []

    for idx,drone_type in enumerate(drone_types):
        label = np.zeros(num_classes)
        label[idx] = 1
        type_folder = os.path.join(data_folder,drone_type)
        if not os.path.exists(type_folder):
            print(f"{type_folder} not exist,skip")
            continue
        
        x_pos = pd.read_csv(os.path.join(type_folder,loca_suffixes[0]),header=None).values.squeeze()
        y_pos = pd.read_csv(os.path.join(type_folder,loca_suffixes[1]),header=None).values.squeeze()           
        z_pos = pd.read_csv(os.path.join(type_folder,loca_suffixes[2]),header=None).values.squeeze()
        position_labels = np.stack([x_pos,y_pos,z_pos],axis=1)
        print(position_labels.shape)
        all_positions.append(position_labels)
        # Read mel spectrogram
        file_list = [f for f in os.listdir(type_folder) if f.startswith('spectrogram2_') and f.endswith('.npy')]
        for selected_file in file_list:
            arr = np.load(os.path.join(type_folder,selected_file))
            all_data.append(arr)
            all_labels.append(label)
    all_data = np.asarray(all_data)
    all_labels = np.asarray(all_labels)
    all_positions = np.vstack(all_positions)
    keep_idx = np.where((all_positions[:,2]>=10)&(all_positions[:,2]<=15))[0]
    X_data = all_data[keep_idx]
    y_classify = all_labels[keep_idx]
    y_location = all_positions[keep_idx]
    print(f"Read completed, Data shape:{X_data.shape}, Category label shape:{y_classify.shape}, Location label shape:{y_location.shape}")
    return X_data,y_classify,y_location


def post_process_data(
    X_data,
    y_classify,
    y_location,
    test_size = 0.2
):
    X_data = np.transpose(X_data,(0,2,3,1))
    
    np.random.seed(42)
    indices = np.random.permutation(X_data.shape[0])
    print(indices.shape)
    X_data = X_data[indices]
    y_classify = y_classify[indices]
    y_location = y_location[indices]

    
    X_train_val_data,X_test_data,y_train_val_classify,y_test_classify,y_train_val_location,y_test_location = train_test_split(X_data,y_classify,y_location,test_size=test_size,shuffle=True,random_state=random_state,stratify=y_classify)
    return X_train_val_data,X_test_data,y_train_val_classify,y_test_classify,y_train_val_location,y_test_location 

X_data,y_classify,y_location = load_spectrogram(data_folder,drone_types,loca_suffixes,num_classes,mic_num)
test_size = 0.2
random_state = 42

X_train_val_data,X_test_data,y_train_val_classify,y_test_classify,y_train_val_location,y_test_location = post_process_data(X_data,y_classify,y_location,test_size=test_size)
t1 = time.time()
elapsed_time = t1-t0
print(f"Time:{elapsed_time:.4f} s")

In [ ]:
# =====================  Establish AudioNet  ================ 
def build_audio_model(input_shape = (64,64,4),feature_dim = 512, num_classes = 4):
    inputs = layers.Input(shape = input_shape)
    # Time convolution
    t1 = layers.Conv2D(16,(3,64),activation='relu',padding = 'valid',data_format = 'channels_last')(inputs)
    t1 = layers.SpatialDropout2D(0.3)(t1)
    t1 = layers.Flatten()(t1)
    t2 = layers.Conv2D(16,(5,64),activation='relu',padding = 'valid',data_format = 'channels_last')(inputs)
    t2 = layers.SpatialDropout2D(0.3)(t2)
    t2 = layers.Flatten()(t2)

    # Frequency convolution
    f1 = layers.Conv2D(16,(64,3),activation='relu',padding = 'valid',data_format = 'channels_last')(inputs)
    f1 = layers.SpatialDropout2D(0.3)(f1)
    f1 = layers.Flatten()(f1)
    f2 = layers.Conv2D(16,(64,5),activation='relu',padding = 'valid',data_format = 'channels_last')(inputs)
    f2 = layers.SpatialDropout2D(0.3)(f2)
    f2 = layers.Flatten()(f2)
    
    x = layers.Concatenate()([t1,t2,f1,f2])
    x = layers.Dense(feature_dim,activation = 'relu')(x)

    # classify
    class_head = layers.Dense(256,activation = 'relu')(x)
    class_head = layers.Dense(128,activation = 'relu')(class_head)
    class_output = layers.Dense(num_classes,activation = 'softmax',name = 'class_output')(class_head)

    # location
    pos_head = layers.Dense(256,activation = 'relu')(x)
    pos_head = layers.Dense(128,activation = 'relu')(pos_head)
    pos_output = layers.Dense(3,activation = 'linear',name='pos_output')(pos_head)
    model = keras.Model(inputs=inputs,outputs = [pos_output,class_output])
    return model

def make_fusion_dataset(X_data,y_classify,y_location,batch_size,shuffle=True):
    N = X_data.shape[0]
    inputs = X_data
    targets = {
        'class_output':y_classify,
        'pos_output':y_location
    }
    ds = tf.data.Dataset.from_tensor_slices((inputs,targets))
    if shuffle:
        ds = ds.shuffle(N)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds
    
def euclidean_distance_loss(y_true,y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    return tf.reduce_mean(tf.norm(y_pred-y_true,axis=-1))

In [ ]:
def save_history_to_csv(history_data, filename,save_dir):
    with open(os.path.join(save_dir, filename), "w", newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(np.mat(history_data).transpose().tolist())


def save_result_fold(save_dir,history,predictions,y_val_classify,y_val_location,main_model,compile_metrics_value):

    train_loss = history.history['loss']
    valid_loss = history.history['val_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(train_loss, 'b', label='Training loss')
    plt.plot(valid_loss, 'r', label='Validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Loss.png'))
    plt.close()

    classification_model_accuracy = history.history['class_output_accuracy']
    val_classification_model_accuracy = history.history['val_class_output_accuracy']
    plt.figure(figsize=(10, 6))
    plt.plot(classification_model_accuracy, 'b', label='Training classification_model_accuracy')
    plt.plot(val_classification_model_accuracy, 'r', label='Validation classification_model_accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('classification_model_accuracy')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'classification_model_accuracy.png'))
    plt.close()

    pos_pred_euclidean_distance_loss = history.history['pos_output_euclidean_distance_loss']
    val_pos_pred_euclidean_distance_loss = history.history['val_pos_output_euclidean_distance_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(pos_pred_euclidean_distance_loss, 'b', label='Training pos_pred_euclidean_distance_loss')
    plt.plot(val_pos_pred_euclidean_distance_loss, 'r', label='Validation pos_pred_euclidean_distance_loss')
    plt.xlabel('Epochs')
    plt.ylabel('pos_pred_euclidean_distance_loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'pos_pred_euclidean_distance_loss.png'))
    plt.close()



    save_history_to_csv(history.history['loss'], 'train_loss.csv',save_dir)
    save_history_to_csv(history.history['val_loss'], 'val_loss.csv',save_dir)

    save_history_to_csv(history.history['class_output_accuracy'], 'Class_Accuracy.csv',save_dir)
    save_history_to_csv(history.history['val_class_output_accuracy'], 'Val_Class_Accuracy.csv',save_dir)

    save_history_to_csv(history.history['pos_output_euclidean_distance_loss'], 'Loca_APE.csv',save_dir)
    save_history_to_csv(history.history['val_pos_output_euclidean_distance_loss'], 'Val_Loca_APE.csv',save_dir)

    y_pred_classes_prob = predictions[1]
    y_pred_classes = np.argmax(y_pred_classes_prob, axis=1)
    y_true_classes = np.argmax(y_val_classify, axis=1)
    y_pred_pos = predictions[0]
    errors = np.linalg.norm(y_pred_pos - y_val_location, axis=1)
    save_history_to_csv(errors, 'test_errors.csv',save_dir)

    APE, rmse = np.mean(errors), np.sqrt(np.mean(errors**2))
    print(f'APE: {APE:.3f}m, RMSE: {rmse:.3f}m, Max: {np.max(errors):.3f}m, Min: {np.min(errors):.3f}m')
    plt.figure(figsize=(10,6))
    plt.plot(errors, label='Position Error')
    plt.xlabel('Sample index')
    plt.ylabel('Error (m)')
    plt.legend()
    plt.title('Test Sample Position Error')
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Test_position_error.png'))
    plt.close()

    conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)
    drone_types = ['Avata','M300','Pham4','Mavic2','Mavic3']
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=drone_types, yticklabels=drone_types)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Confusion Matrix')
    plt.savefig(os.path.join(save_dir, 'Confusion_Matrix.png'))
    plt.close()

    classification_rep = classification_report(y_true_classes, y_pred_classes, target_names=drone_types)
    print("Classification_Report:\n",classification_rep)
    with open(os.path.join(save_dir, 'Classification_Report.txt'),"w") as f:
        f.write(classification_rep)
    

    pd.DataFrame({'cls':[compile_metrics_value],'mae':[APE],'rmse':[rmse]}).to_csv(os.path.join(save_dir,'test_position_error.csv'))
    print("Save")
    output_path_model = os.path.join(save_dir,'my_main_model.keras')
    main_model.save(output_path_model)
    

    filename = 'main_model_summary.txt'
    file_path = os.path.join(save_dir,filename)
    with open(file_path,'w') as f:
        main_model.summary(print_fn = lambda x: f.write(x+'\n'))

    return compile_metrics_value,APE


In [18]:
def get_model(num_classes):
    tf.keras.backend.clear_session()
    model = build_audio_model(input_shape=(64,64,4),feature_dim=512,num_classes=num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={
            'class_output':tf.keras.losses.CategoricalCrossentropy(),
            'pos_output':euclidean_distance_loss
        },
        loss_weights= {
            'class_output':1.0,
            'pos_output':1.0
        },
        metrics={
            'class_output':['accuracy'],
            'pos_output':[euclidean_distance_loss]
        }
    )
    return model
def train_with_history(model, X_train_dict, X_val_dict, epochs, batch_size, verbose=1):
    result = model.fit(
        X_train_dict, validation_data=X_val_dict,
        epochs=epochs, batch_size=batch_size,
        verbose=verbose
    )
    return result

In [ ]:

nb_epoch = 1000
batch_size = 64
n_splits = 10


t0 = time.time()
loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
base_path = f"Public_Audio_Compare_numclasses_5/Batchsize_{batch_size}_Nbepoch_{nb_epoch}_height_between_10_and_15m/{loca}/"
os.makedirs(base_path,exist_ok = True)

y_trainval_cls = np.argmax(y_train_val_classify,axis=1)
print('y_trainval_cls former 20 items:',y_trainval_cls[:20])
print('Category distribution:',np.bincount(y_trainval_cls))

skf = StratifiedKFold(n_splits = n_splits,shuffle=True,random_state=random_state)
acc_list = []
ape_list = []

print(f"\n=== {n_splits} fold cross-validation ====")

for fold,(train_idx,val_idx) in enumerate(skf.split(X_train_val_data,y_trainval_cls),1):
    
    print(f"\n==== Fold {fold}/{n_splits} ====")
    fold_loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
    fold_path = f"{fold_loca}-fold{fold}"
    fold_dir = os.path.join(base_path,fold_path)
    os.makedirs(fold_dir,exist_ok = True)

    X_train_data = X_train_val_data[train_idx]
    X_val_data = X_train_val_data[val_idx]
    y_train_classify = y_train_val_classify[train_idx]
    y_val_classify = y_train_val_classify[val_idx]
    y_train_location = y_train_val_location[train_idx]
    y_val_location = y_train_val_location[val_idx]

    print(f"Train set:{X_train_data.shape[0]},Validation set:{X_val_data.shape[0]}")

    main_model = get_model(num_classes)
    X_train_dict = make_fusion_dataset(X_train_data,y_train_classify,y_train_location,batch_size=batch_size)
    X_val_dict = make_fusion_dataset(X_val_data,y_val_classify,y_val_location,batch_size=batch_size,shuffle=False)

    history = train_with_history(main_model,X_train_dict,X_val_dict,nb_epoch,batch_size,verbose=0)
    
    results = main_model.evaluate(X_val_dict,verbose=0)
    
    metrics_dict = dict(zip(main_model.metrics_names, results))
    compile_metrics_value = metrics_dict['class_output_accuracy']

    predictions = main_model.predict(X_val_dict)

    temp_acc,temp_ape = save_result_fold(fold_dir,history,predictions,y_val_classify,y_val_location,main_model,compile_metrics_value)
    acc_list.append(temp_acc)
    ape_list.append(temp_ape)

    msg = (f"Fold {fold} Acc: {temp_acc:.4f} Ape: {temp_ape:.4f}\n")
    with open(os.path.join(fold_dir,"fold_result.txt"),"w",encoding = "utf-8") as f:
        f.write(msg)

      
t1=time.time()
elapsed_time = t1-t0
print(f"Time: {elapsed_time:.4f} s")

In [ ]:

print("\n==== ALL-fold Results ====")
acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)
ape_mean, ape_std = np.mean(ape_list), np.std(ape_list)
print(f"Acc: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"APE: {ape_mean:.4f} ± {ape_std:.4f}")

summary_msg = (
    "==== ALL-fold Results ====\n"
    f"Acc: {acc_mean:.4f} ± {acc_std:.4f}\n"
    f"Ape: {ape_mean:.4f} ± {ape_std:.4f}"
)
summary_path = os.path.join(base_path, "results_summary.txt")


with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_msg)

print(f"Save to: {summary_path}")


In [ ]:
# ======================= 2. Retrain the final model and evaluate on the test set =====================
print("\nRetrain the final model and evaluate on the test set")

final_model = get_model(num_classes)
X_train_val_dict = make_fusion_dataset(X_train_val_data,y_train_val_classify,y_train_val_location,batch_size=batch_size)
X_test_dict = make_fusion_dataset(X_test_data,y_test_classify,y_test_location,batch_size=batch_size,shuffle=False)

final_history = train_with_history(final_model, X_train_val_dict, X_train_val_dict,nb_epoch, batch_size, verbose=0)
test_result = final_model.evaluate(X_test_dict, verbose=2)
test_metrics_dict = dict(zip(final_model.metrics_names, test_result))
test_compile_metrics_value = test_metrics_dict['class_output_accuracy']
test_dir = os.path.join(base_path, "test")
os.makedirs(test_dir, exist_ok=True)
test_acc,test_ape = save_result_fold(test_dir, final_history, final_model.predict(X_test_dict), y_test_classify, y_test_location,final_model,test_compile_metrics_value)

summary_msg_test = (
    "==== Test Results After ALL Fold ====\n"
    f"Test set Acc: {test_acc:.4f}\n"
    f"Test set Ape: {test_ape:.4f}"
)
summary_path = os.path.join(test_dir, "test_set_results_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_msg_test)

print(f"Save to: {summary_path}")

